In [ ]:
import requests
from datetime import datetime, timedelta
import pytz
from geopy.geocoders import Nominatim

KP_URL = "https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json"
PLASMA_URL = "https://services.swpc.noaa.gov/products/solar-wind/plasma-1-day.json"
MAG_URL = "https://services.swpc.noaa.gov/products/solar-wind/mag-1-day.json"


def fetch_kp():
    data = requests.get(KP_URL, timeout=10).json()
    return float(data[-1][1])

def fetch_solar_wind():
    plasma = requests.get(PLASMA_URL, timeout=10).json()
    mag = requests.get(MAG_URL, timeout=10).json()
    return {
        "speed": float(plasma[-1][2]),   # km/s
        "bz": float(mag[-1][6])          # nT
    }

def get_location_coords(place_name):
    geolocator = Nominatim(user_agent="aurora_model")
    location = geolocator.geocode(place_name)
    if not location:
        raise ValueError("Location not found")
    return location.latitude, location.longitude

def auroral_oval_latitude(kp):
    
    return 67 - kp * 2

def night_factor(local_hour):
    
    if 22 <= local_hour or local_hour <= 4:
        return 1.0
    if 20 <= local_hour <= 22 or 4 <= local_hour <= 6:
        return 0.6
    return 0.2

def aurora_probability(lat, kp, speed, bz, hour):
    oval_lat = auroral_oval_latitude(kp)
    lat_distance = abs(lat - oval_lat)

    if lat_distance > 15:
        base = 0.05
    else:
        base = max(0, 1 - lat_distance / 15)

    # Solar wind 
    if speed > 600:
        base += 0.15
    if bz < -5:
        base += 0.2
    if bz < -10:
        base += 0.2

    base *= night_factor(hour)
    return min(base, 1.0) * 100


def aurora_forecast():
    # Ask the user for location input
    location_name = input("Enter your location (city or place name): ").strip()
    try:
        lat, lon = get_location_coords(location_name)
    except ValueError as e:
        print(e)
        return

    kp = fetch_kp()
    solar = fetch_solar_wind()

    tz = pytz.utc
    now = datetime.now(tz)

    print(f"\nLocation: {location_name}")
    print(f"Latitude: {lat:.2f}°")
    print(f"Kp Index: {kp}")
    print(f"Solar wind speed: {solar['speed']:.0f} km/s")
    print(f"IMF Bz: {solar['bz']:.1f} nT\n")

    print("🕒 Aurora Probability (UTC)")
    print("──────────────────────────")

    # Forecast next 8 hours, every 2 hours
    for h in range(0, 8, 2):
        t = now + timedelta(hours=h)
        hour = t.hour
        prob = aurora_probability(lat, kp, solar["speed"], solar["bz"], hour)
        print(f"{t.strftime('%H:%M')} → {prob:.1f}%")



if __name__ == "__main__":
    aurora_forecast()



Location: Chicago
Latitude: 41.88°
Kp Index: 3.0
Solar wind speed: 520 km/s
IMF Bz: 8.2 nT

🕒 Aurora Probability (UTC)
──────────────────────────
01:21 → 5.0%
03:21 → 5.0%
05:21 → 3.0%
07:21 → 1.0%


In [ ]:


import requests
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


KP_URL = "https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json"
PLASMA_URL = "https://services.swpc.noaa.gov/products/solar-wind/plasma-1-day.json"
MAG_URL = "https://services.swpc.noaa.gov/products/solar-wind/mag-1-day.json"
OVATION_URL = "https://services.swpc.noaa.gov/json/ovation_aurora_latest.json"


def fetch_latest_kp():
    data = requests.get(KP_URL, timeout=10).json()
    return float(data[-1][1])

def fetch_latest_solar_wind():
    plasma = requests.get(PLASMA_URL, timeout=10).json()
    mag = requests.get(MAG_URL, timeout=10).json()
    plasma_latest = plasma[-1]
    mag_latest = mag[-1]
    return {
        "speed": float(plasma_latest[2]),     # km/s
        "density": float(plasma_latest[1]),   # p/cm^3
        "bz": float(mag_latest[6]),           # nT
        "bt": float(mag_latest[1])            # nT
    }

def fetch_real_aurora(lat, lon):
    """
    Get actual aurora probability from NOAA OVATION for a specific lat/lon.
    OVATION provides a grid: find nearest point.
    """
    data = requests.get(OVATION_URL, timeout=10).json()
    
    # aurora grid: [lat, lon, probability]
    if "aurora" not in data:
        print("Warning: No 'aurora' field in OVATION data, using max probability instead.")
        return data.get("probability", [0])[0]
    
    aurora_grid = data["aurora"]
    
    # Find nearest latitude/longitude 
    distances = [ (abs(row[0]-lat)+abs(row[1]-lon), row[2]) for row in aurora_grid if len(row) >=3 ]
    
    if not distances:
        return 0
    
    nearest_prob = max(distances, key=lambda x: -x[1])[1]  # Take max probability nearby
    return nearest_prob

def kp_to_label(kp):
    if kp <= 2:
        return "weak"
    elif kp <= 5:
        return "moderate"
    else:
        return "strong"

np.random.seed(42)
samples = []

for _ in range(500):
    kp = np.random.randint(0, 9)
    sample = {
        "kp": kp,
        "speed": np.random.normal(450 + kp*30, 50),
        "density": np.random.normal(5 + kp*0.5, 1),
        "bz": np.random.normal(-kp, 2),
        "bt": np.random.normal(5 + kp, 1),
        "label": kp_to_label(kp)
    }
    samples.append(sample)

df = pd.DataFrame(samples)

X = df[["speed","density","bz","bt"]]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)


print("Enter the coordinates for aurora prediction:")
lat = float(input("Latitude (e.g., 64.13 for Reykjavik): "))
lon = float(input("Longitude (e.g., -21.82 for Reykjavik): "))

kp_now = fetch_latest_kp()
solar_now = fetch_latest_solar_wind()
X_live = pd.DataFrame([solar_now])
X_live_scaled = scaler.transform(X_live)

# ML prediction
probs = model.predict_proba(X_live_scaled)[0]
classes = model.classes_
class_probs = dict(zip(classes, probs))

# Physics-based interpretation
formation_pct = class_probs["weak"]*0.3 + class_probs["moderate"]*0.7 + class_probs["strong"]*0.95
formation_pct *= 100

# Visibility approximation
vis = 0.4
if solar_now["speed"] > 500: vis += 0.2
if solar_now["speed"] > 650: vis += 0.2
if solar_now["bz"] < -5: vis += 0.2
if solar_now["bz"] < -10: vis += 0.3
visibility_pct = min(vis,1.0)*100

# Colour estimation
green = min(0.8, 0.4 + kp_now*0.05)
red = min(0.7, max(0, (kp_now-4)*0.08))
purple = min(0.6, max(0, (-solar_now["bz"]-5)*0.07))
total = green+red+purple
colour_probs = {"green":green/total*100,"red":red/total*100,"purple":purple/total*100}

latitude_reach = max(45, 67 - kp_now*2)


noaa_prob = fetch_real_aurora(lat, lon)
abs_error = abs(formation_pct - noaa_prob)
rel_error = abs_error / max(noaa_prob,1) * 100

# results
print("\nAURORA PREDICTION FOR LAT:", lat, "LON:", lon)
print("────────────────────────────")
print(f"Kp Index: {kp_now}")
print(f"Solar wind speed: {solar_now['speed']:.1f} km/s")
print(f"IMF Bz: {solar_now['bz']:.1f} nT\n")

print("Aurora Class Probabilities:")
for k,v in class_probs.items():
    print(f"  {k.capitalize():8s}: {v*100:.1f}%")

print(f"\nAurora Formation Probability: {formation_pct:.1f}%")
print(f"Visibility Probability:       {visibility_pct:.1f}%")
print("\nExpected Colour Composition:")
for k,v in colour_probs.items():
    print(f"  {k.capitalize():6s}: {v:.1f}%")
print(f"Estimated Latitude Reach: ~{latitude_reach:.1f}°")





Enter the coordinates for aurora prediction:

AURORA PREDICTION FOR LAT: 55.32 LON: -20.43
────────────────────────────
Kp Index: 3.0
Solar wind speed: 524.7 km/s
IMF Bz: 8.2 nT

Aurora Class Probabilities:
  Moderate: 0.5%
  Strong  : 0.0%
  Weak    : 99.5%

Aurora Formation Probability: 30.2%
Visibility Probability:       60.0%

Expected Colour Composition:
  Green : 100.0%
  Red   : 0.0%
  Purple: 0.0%
Estimated Latitude Reach: ~61.0°
